# Universal Bronze Ingestion — DataPulse / Customer-Sales project

One notebook, invoked once per dataset using the `dataset_name` job parameter.

Preserves the shared 32-column Delta file manifest and the existing per-file
lifecycle: NEW / FAILED / SUCCESS -> PROCESSING -> SUCCESS or FAILED.
Supports INCREMENTAL, BACKFILL and FULL_LOAD selection, SHA-256 source checks,
conditional claims, file-level Bronze replacement, row-count reconciliation,
and manifest finalization.



In [0]:

# CELL 1 — UNIVERSAL BRONZE CONFIGURATION, WIDGETS, AND LOGGING

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import StructType
from datetime import datetime,timezone, timedelta
import logging
import sys
import re
import uuid
import hashlib


# 1. Configure logger
LOGGER_NAME = "universal_bronze"
logger = logging.getLogger(LOGGER_NAME)
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    ))
    logger.addHandler(handler)

logger.info("Universal Bronze Cell 1 started.")

# 2. Storage configuration
STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@"
    f"{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

MANIFEST_PATH = BASE_PATH + "metadata/file_manifest/"
SCHEMA_CONFIG_PATH = BASE_PATH + "metadata/schema_config/"
LEASE_MINUTES = 10



# dbutils.widgets.removeAll()

dbutils.widgets.text(
    "dataset_name", "google_ads_campaigns", "Dataset Name"
)

dbutils.widgets.text(
    "source_system", "google_ads", "Source System (optional)"
)

dbutils.widgets.dropdown(
    "run_mode",
    "INCREMENTAL",
    ["INCREMENTAL", "BACKFILL", "FULL_LOAD"],
    "Run Mode"
)

dbutils.widgets.text(
    "backfill_start_date", "", "Backfill Start Date (YYYY-MM-DD)"
)

dbutils.widgets.text(
    "backfill_end_date", "", "Backfill End Date (YYYY-MM-DD)"
)

dbutils.widgets.dropdown(
    "reprocess_successful",
    "false",
    ["false", "true"],
    "Reprocess Successful Files?"
)



# 4. Read widget values
DATASET_NAME = (
    dbutils.widgets.get("dataset_name").strip().lower()
)

SOURCE_SYSTEM_INPUT = (
    dbutils.widgets.get("source_system").strip()
)
RUN_MODE = (
    dbutils.widgets.get("run_mode").strip().upper()
)
BACKFILL_START_DATE = (
    dbutils.widgets.get("backfill_start_date").strip()
)
BACKFILL_END_DATE = (
    dbutils.widgets.get("backfill_end_date").strip()
)
REPROCESS_SUCCESSFUL = (
    dbutils.widgets.get("reprocess_successful").strip().lower()
    == "true"
)


# 5. Validate required parameters
if not DATASET_NAME:
    raise ValueError(
        "Enter dataset_name, for example google_ads_campaigns."
    )

if not re.fullmatch(r"[a-z][a-z0-9_]*", DATASET_NAME):
    raise ValueError(
        "dataset_name must contain lowercase letters, digits, "
        "and underscores."
    )

if RUN_MODE not in {"INCREMENTAL", "BACKFILL", "FULL_LOAD"}:
    raise ValueError(f"Unsupported run_mode: {RUN_MODE}")

if RUN_MODE == "BACKFILL":
    if not BACKFILL_START_DATE or not BACKFILL_END_DATE:
        raise ValueError(
            "BACKFILL requires both backfill_start_date "
            "and backfill_end_date."
        )

    start_date = datetime.strptime(
        BACKFILL_START_DATE, "%Y-%m-%d"
    ).date()

    end_date = datetime.strptime(
        BACKFILL_END_DATE, "%Y-%m-%d"
    ).date()

    if start_date > end_date:
        raise ValueError(
            "Backfill start date must be <= end date."
        )



BRONZE_PATH = BASE_PATH + f"bronze/{DATASET_NAME}/"

# 7. Load exactly one active schema for the selected dataset
if not DeltaTable.isDeltaTable(spark, SCHEMA_CONFIG_PATH):
    raise RuntimeError(
        f"Schema configuration Delta table not found: "
        f"{SCHEMA_CONFIG_PATH}"
    )

schema_config_df = (
    spark.read.format("delta").load(SCHEMA_CONFIG_PATH)
)

active_schema_rows = (
    schema_config_df
    .filter(
        (F.lower(F.trim(F.col("dataset_name"))) == DATASET_NAME)
        & (F.col("is_active") == True)
    )
    .select(
        "source_system",
        "schema_ddl",
        "schema_version"
    )
    .collect()
)

if len(active_schema_rows) != 1:
    raise ValueError(
        f"Expected exactly one active schema for "
        f"{DATASET_NAME}; found {len(active_schema_rows)}. "
        "Register or activate the dataset schema before running."
    )

SCHEMA_SOURCE_SYSTEM = (
    active_schema_rows[0]["source_system"]
)

SCHEMA_DDL = active_schema_rows[0]["schema_ddl"]
SCHEMA_VERSION = active_schema_rows[0]["schema_version"]

if not SCHEMA_DDL or not SCHEMA_DDL.strip():
    raise ValueError(
        f"schema_ddl is empty for {DATASET_NAME}."
    )

SOURCE_SCHEMA = StructType.fromDDL(SCHEMA_DDL)

SOURCE_SYSTEM = (
    SOURCE_SYSTEM_INPUT or SCHEMA_SOURCE_SYSTEM
)

if not SOURCE_SYSTEM:
    raise ValueError(
        f"Source system is missing for {DATASET_NAME}."
    )



# 8. Standard ingestion metadata for new datasets
# Dataset-specific legacy metadata profiles should be added
# explicitly if existing Bronze schemas must be preserved.
METADATA_PROFILE = "standard"
INGESTED_AT_COL = "ingestion_timestamp"
SOURCE_SYSTEM_COL = "source_system"
SOURCE_FILE_COL = "source_file"
SOURCE_PATH_COL = None
DATASET_COL = None
RUN_ID_COL = "run_id"
BATCH_ID_COL = "batch_id"

# 9. Log resolved configuration
logger.info(
    "Dataset=%s | source_system=%s | schema_version=%s | schema_fields=%s",
    DATASET_NAME,
    SOURCE_SYSTEM,
    SCHEMA_VERSION,
    len(SOURCE_SCHEMA.fields)
)

logger.info("Run mode: %s", RUN_MODE)
logger.info("Bronze path: %s", BRONZE_PATH)
logger.info("Manifest path: %s", MANIFEST_PATH)
logger.info("Schema config path: %s", SCHEMA_CONFIG_PATH)
logger.info("Universal Bronze Cell 1 completed.")


In [0]:

# =========================================================
# CELL 2 — UNIVERSAL FILE DISCOVERY AND SHA-256
# READ-ONLY: No manifest or Bronze writes
# =========================================================

from pyspark.sql.types import (
    StructType, StructField, StringType, LongType
)
from pyspark.sql import functions as F               
import re                                             

logger.info("Universal Bronze Cell 2 started.")         

# ---------------------------------------------------------
# 1. Validate parameters from Cell 1
# ---------------------------------------------------------

if not DATASET_NAME or not DATASET_NAME.strip():
    raise ValueError("DATASET_NAME cannot be empty.")

if not SOURCE_SYSTEM or not SOURCE_SYSTEM.strip():
    raise ValueError("SOURCE_SYSTEM cannot be empty.")

DATASET_NAME = DATASET_NAME.strip().lower()
SOURCE_SYSTEM = SOURCE_SYSTEM.strip().lower()

LANDING_SUBPATH = (
    f"{SOURCE_SYSTEM}/{DATASET_NAME}"
)


LANDING_PATH = (
    BASE_PATH.rstrip("/")
    + "/landing/"
    + LANDING_SUBPATH
    + "/"
)

logger.info("Dataset: %s", DATASET_NAME)
logger.info("Source system: %s", SOURCE_SYSTEM)
logger.info("Landing path: %s", LANDING_PATH)

# ---------------------------------------------------------
# 2. Recursively discover files
# ---------------------------------------------------------

def list_files_recursive(path):
    found = []

    for item in dbutils.fs.ls(path):
        if item.isDir():
            found.extend(list_files_recursive(item.path))
        else:
            found.append(
                (item.name, item.path, int(item.size))
            )
    return found 


try:
    discovered = list_files_recursive(LANDING_PATH)
    print(discovered)

except Exception as exc:
    raise RuntimeError(
        f"Unable to list landing path: {LANDING_PATH}. "
        f"Verify the path and storage permissions. "
        f"Dataset={DATASET_NAME}, source_system={SOURCE_SYSTEM}"
    ) from exc

landing_schema = StructType([
    StructField("file_name", StringType(), False),
    StructField("file_path", StringType(), False),
    StructField("file_size_bytes", LongType(), False),
])

landing_df = spark.createDataFrame(
    discovered,
    schema=landing_schema
)

logger.info("Total files discovered: %s", len(discovered))

# ---------------------------------------------------------
# 3. Validate filenames and extract data dates
# ---------------------------------------------------------
# Expected format:
# <FILENAME_PREFIX>_YYYYMMDD.json

dated_df = (
    landing_df
    .withColumn(
        "date_string",
        F.regexp_extract(
            F.col("file_name"),
            r"(\d{8})",
            1
        ))
    .withColumn("data_date",
               F.to_date(F.regexp_extract(
            F.col("file_name"),
            r"(\d{8})",
            1
        ),"yyyyMMdd"))
    .withColumn(
        "filename_status",
        F.when(
            F.col("date_string").isNull(),
            F.lit("INVALID_FILENAME")
        )
        .when(
            F.col("data_date").isNull()
            | (
                F.date_format(
                    F.col("data_date"),
                    "yyyyMMdd"
                ) != F.col("date_string")
            ),
            F.lit("INVALID_DATE")
        )
        .otherwise(F.lit("VALID"))
    )
)

invalid_df = dated_df.filter(
    F.col("filename_status") != "VALID"
)

invalid_count = invalid_df.count()

if invalid_count > 0:
    display(
        invalid_df.select(
            "file_name",
            "file_path",
            "filename_status"
        )
    )

    raise ValueError(
        f"Found {invalid_count} invalid file(s) under "
        f"{LANDING_PATH}. Expected filenames like "
        "Fix or separate unexpected files before continuing."
    )
display(dated_df)
# ---------------------------------------------------------
# 4. Calculate SHA-256 hashes
# ---------------------------------------------------------

if discovered:
    file_paths = [item[1] for item in discovered]

    binary_df = (
        spark.read
        .format("binaryFile")
        .load(file_paths)
        .select(
            F.col("path").alias("file_path"),
            F.sha2(
                F.col("content"),
                256
            ).alias("actual_content_hash")
        )
    )

    files_df = (
        dated_df
        .filter(F.col("filename_status") == "VALID")
        .join(
            binary_df,
            on="file_path",
            how="left"
        )
    )

    unhashed_count = files_df.filter(
        F.col("actual_content_hash").isNull()
    ).count()

    if unhashed_count > 0:
        raise RuntimeError(
            f"SHA-256 could not be calculated for "
            f"{unhashed_count} discovered file(s)."
        )

else:
    files_df = (
        dated_df
        .filter(F.col("filename_status") == "VALID")
        .withColumn(
            "actual_content_hash",
            F.lit(None).cast("string")
        )
    )

# ---------------------------------------------------------
# 5. Display discovery results
# ---------------------------------------------------------

display(
    files_df
)

print("Dataset:", DATASET_NAME)
print("Source system:", SOURCE_SYSTEM)
print("Landing path:", LANDING_PATH)
print("Discovered valid files:", files_df.count())

logger.info("Universal Bronze Cell 2 completed.")  


In [0]:

# CELL 3 — RECONCILE AND SELECT FILES (READ-ONLY)

# 1. Read the existing manifest
if not DeltaTable.isDeltaTable(spark, MANIFEST_PATH):
    raise RuntimeError(
        f"Manifest Delta table not found: {MANIFEST_PATH}"
    )

manifest_all_df = (
    spark.read.format("delta").load(MANIFEST_PATH)
)

# 2. Keep only records for the selected dataset
manifest_df = (
    manifest_all_df
    .filter(F.col("dataset") == DATASET_NAME)
    .select(
        "dataset",
        "file_name",
        "status",
        "content_hash",
        "data_date",
        "retry_count",
        "attempt_count",
        "run_id"
    )
)

# 3. Check for duplicate manifest keys within this dataset
duplicates = (
    manifest_df
    .groupBy("dataset", "file_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicates > 0:
    raise ValueError(
        f"Found {duplicates} duplicate manifest keys for "
        f"dataset={DATASET_NAME}."
    )

# 4. Reconcile landing files with manifest records
reconciliation_df = (
    files_df.alias("l")
    .join(
        manifest_df.alias("m"),
        (F.col("l.file_name") == F.col("m.file_name"))
        & (F.lit(DATASET_NAME) == F.col("m.dataset")),
        "left"
    )
    .select(
        F.lit(DATASET_NAME).alias("dataset"),
        F.col("l.file_name"),
        F.col("l.file_path"),
        F.col("l.file_size_bytes"),
        F.col("l.data_date").alias("landing_data_date"),
        F.col("l.actual_content_hash"),
        F.col("m.status").alias("manifest_status"),
        F.col("m.content_hash").alias("manifest_hash"),
        F.col("m.data_date").alias("manifest_data_date"),
        F.col("m.retry_count"),
        F.col("m.attempt_count"),
        F.col("m.run_id").alias("manifest_run_id")
    )
    .withColumn(
        "effective_data_date",
        F.coalesce("manifest_data_date", "landing_data_date")
    )
    .withColumn(
        "selection_reason",
        F.when(
            F.col("manifest_status") == "PROCESSING",
            "SKIP_PROCESSING"
        )
        .when(
            F.col("manifest_status").isNull(),
            "REGISTER_REQUIRED"
        )
        .when(
            F.col("manifest_hash").isNull(),
            "REVIEW_REQUIRED"
        )
        .when(
            F.col("manifest_hash") != F.col("actual_content_hash"),
            "CONTENT_CHANGED"
        )
        .when(
            F.col("manifest_status").isin("NEW", "FAILED"),
            "ELIGIBLE"
        )
        .when(
            F.col("manifest_status") == "SUCCESS",
            "SUCCESS_REPROCESS_CANDIDATE"
        )
        .otherwise("REVIEW_REQUIRED")
    )
)

# 5. Select candidates using the existing run-mode rules
base_reasons = [
    "REGISTER_REQUIRED",
    "ELIGIBLE",
    "CONTENT_CHANGED"
]

if RUN_MODE == "INCREMENTAL":
    selected_df = reconciliation_df.filter(
        F.col("selection_reason").isin(base_reasons)
    )

elif RUN_MODE == "BACKFILL":
    reasons = base_reasons.copy()

    if REPROCESS_SUCCESSFUL:
        reasons.append("SUCCESS_REPROCESS_CANDIDATE")

    selected_df = reconciliation_df.filter(
        F.col("effective_data_date").between(
            F.to_date(F.lit(BACKFILL_START_DATE)),
            F.to_date(F.lit(BACKFILL_END_DATE))
        )
        & F.col("selection_reason").isin(reasons)
    )

elif RUN_MODE == "FULL_LOAD":
    # Preserves the original behavior:
    # successful files are included regardless of REPROCESS_SUCCESSFUL.
    selected_df = reconciliation_df.filter(
        F.col("selection_reason").isin(
            base_reasons + ["SUCCESS_REPROCESS_CANDIDATE"]
        )
    )

else:
    raise ValueError(f"Unsupported run mode: {RUN_MODE}")

# 6. Review reconciliation and selection
display(
    reconciliation_df.select(
        "dataset",
        "file_name",
        "effective_data_date",
        "manifest_status",
        "selection_reason"
    ).orderBy("effective_data_date", "file_name")
)
display(selected_df)

print("Dataset:", DATASET_NAME)
print("Source system:", SOURCE_SYSTEM)
print("Discovered files:", reconciliation_df.count())
print("Selected candidates:", selected_df.count())


In [0]:

# CELL 4 — UNIVERSAL PREFLIGHT, SAFETY GATE, AND DRY-RUN PLAN
# READ-ONLY: does not register files, claim files, or write Bronze data.

logger.info("Universal Bronze Cell 4 started.")

# 1. Validate the shared manifest contract
manifest_required = {
    "dataset", "file_name", "file_path", "batch_id", "status",
    "first_seen", "processed_at", "run_id", "retry_count",
    "error_message", "data_date", "file_size_bytes",
    "source_modified_at", "content_hash", "last_seen_at",
    "discovery_run_id", "claimed_at", "lease_expires_at",
    "last_attempt_at", "attempt_count", "last_success_at",
    "status_updated_at", "source_record_count",
    "written_record_count", "rejected_record_count",
    "processed_content_hash", "schema_fingerprint",
    "target_path", "target_delta_version",
    "processing_duration_seconds", "error_type", "last_error_at"
}

manifest_missing = sorted(
    manifest_required - set(manifest_all_df.columns)
)

if manifest_missing:
    raise ValueError(
        f"Manifest schema is missing required columns: "
        f"{manifest_missing}"
    )

logger.info("Manifest required columns: PASS")

# 2. Validate the expected Bronze schema dynamically
# Use source fields from schema_config plus standard metadata.
expected_bronze_columns = set(SOURCE_SCHEMA.fieldNames()) | {
    INGESTED_AT_COL,
    SOURCE_SYSTEM_COL,
    SOURCE_FILE_COL,
    RUN_ID_COL,
    BATCH_ID_COL
}

bronze_exists = DeltaTable.isDeltaTable(spark, BRONZE_PATH)

if bronze_exists:
    bronze_df_existing = (
        spark.read.format("delta").load(BRONZE_PATH)
    )

    bronze_missing = sorted(
        expected_bronze_columns - set(bronze_df_existing.columns)
    )

    if bronze_missing:
        raise ValueError(
            f"Existing Bronze table is missing expected columns "
            f"for dataset {DATASET_NAME}: {bronze_missing}"
        )

    logger.info("Existing Bronze schema: PASS")

else:
    logger.info(
        "Bronze Delta table does not exist yet; "
        "first-write schema validation will occur during ingestion."
    )

# 3. Validate selected candidates
selected_count = selected_df.count()

duplicate_selected = (
    selected_df
    .groupBy("file_path")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

bad_hashes = selected_df.filter(
    F.col("actual_content_hash").isNull()
    | ~F.col("actual_content_hash").rlike("^[0-9a-fA-F]{64}$")
).count()

bad_dates = selected_df.filter(
    F.col("effective_data_date").isNull()
).count()

processing_selected = selected_df.filter(
    F.col("manifest_status") == "PROCESSING"
).count()

review_selected = selected_df.filter(
    F.col("selection_reason") == "REVIEW_REQUIRED"
).count()

if duplicate_selected:
    raise ValueError(
        f"Found {duplicate_selected} duplicate selected file paths."
    )

if bad_hashes:
    raise ValueError(
        f"Found {bad_hashes} selected files with invalid SHA-256 hashes."
    )

if bad_dates:
    raise ValueError(
        f"Found {bad_dates} selected files without an effective data date."
    )

if processing_selected:
    raise ValueError(
        f"Found {processing_selected} selected files already marked PROCESSING."
    )

if review_selected:
    raise ValueError(
        f"Found {review_selected} selected files requiring manual review."
    )

# 4. Display the dry-run processing plan
display(
    selected_df.select(
        "file_name",
        "file_path",
        "effective_data_date",
        "manifest_status",
        "selection_reason",
        "actual_content_hash"
    ).orderBy("effective_data_date", "file_name")
)

print("Preflight PASS")
print("Dataset:", DATASET_NAME)
print("Source system:", SOURCE_SYSTEM)
print("Run mode:", RUN_MODE)
print("Selected candidates:", selected_count)
print("Manifest required columns: PASS")
print(
    "Bronze validation:",
    "EXISTING TABLE PASS" if bronze_exists
    else "FIRST WRITE PENDING"
)
print("Hash, date, duplicate and status checks: PASS")
print("Preflight passed. Selected files are ready for the next stage.")

logger.info("Universal Bronze Cell 4 completed.")


In [0]:
# manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)
# manifest_schema = spark.read.format("delta").load(MANIFEST_PATH).schema
#print(manifest_schema)
# for x in manifest_schema.fields:
#     print(x.name,x.dataType,x.nullable)
# manifest_columns=[x.name for x in manifest_schema.fields]
# print(manifest_columns)

In [0]:
from pyspark.sql import functions as F

new_files_df = (
    reconciliation_df.alias("s")
    .join(
        manifest_df.select("dataset", "file_name").alias("m"),
        on=["dataset", "file_name"],
        how="left_anti"
    )
    .filter(F.col("selection_reason") == "REGISTER_REQUIRED")
    .select(
        F.col("dataset"),
        F.col("file_name"),
        F.col("file_path"),
        F.lit(None).cast("string").alias("batch_id"),
        F.lit("NEW").alias("status"),
        F.current_timestamp().alias("first_seen"),
        F.lit(None).cast("timestamp").alias("processed_at"),
        F.lit(None).cast("string").alias("run_id"),
        F.lit(0).cast("int").alias("retry_count"),
        F.lit(None).cast("string").alias("error_message"),
        F.col("effective_data_date").alias("data_date"),
        F.col("file_size_bytes"),
        F.lit(None).cast("timestamp").alias("source_modified_at"),
        F.col("actual_content_hash").alias("content_hash"),
        F.current_timestamp().alias("last_seen_at"),
        F.lit(run_id).alias("discovery_run_id"),
        F.lit(None).cast("timestamp").alias("claimed_at"),
        F.lit(None).cast("timestamp").alias("lease_expires_at"),
        F.lit(None).cast("timestamp").alias("last_attempt_at"),
        F.lit(0).cast("int").alias("attempt_count"),
        F.lit(None).cast("timestamp").alias("last_success_at"),
        F.current_timestamp().alias("status_updated_at"),
        F.lit(None).cast("long").alias("source_record_count"),
        F.lit(None).cast("long").alias("written_record_count"),
        F.lit(None).cast("long").alias("rejected_record_count"),
        F.lit(None).cast("string").alias("processed_content_hash"),
        F.lit(SCHEMA_FINGERPRINT).alias("schema_fingerprint"),
        F.lit(None).cast("string").alias("target_path"),
        F.lit(None).cast("long").alias("target_delta_version"),
        F.lit(None).cast("double").alias("processing_duration_seconds"),
        F.lit(None).cast("string").alias("error_type"),
        F.lit(None).cast("timestamp").alias("last_error_at")
    )
)

display(new_files_df)
from delta.tables import DeltaTable

manifest_delta = DeltaTable.forPath(
    spark,
    MANIFEST_PATH
)

(
    manifest_delta.alias("t")
    .merge(
        new_files_df.alias("s"),
        """
        t.dataset = s.dataset
        AND t.file_name = s.file_name
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("New files successfully registered in manifest.")

In [0]:
logger.info("Universal Bronze Cell 5 started.")

# ---------------------------------------------------------
# 1. Validate configuration and prepare manifest
# ---------------------------------------------------------

if not selected_df.columns:
    raise RuntimeError("selected_df has no columns. Run Cells 1–4 first.")

required_selected_columns = {
    "file_name",
    "file_path",
    "file_size_bytes",
    "effective_data_date",
    "actual_content_hash",
    "selection_reason",
}

missing_selected_columns = (
    required_selected_columns - set(selected_df.columns)
)

if missing_selected_columns:
    raise ValueError(
        f"selected_df is missing columns: {sorted(missing_selected_columns)}"
    )

if not DeltaTable.isDeltaTable(spark, MANIFEST_PATH):
    raise RuntimeError(f"Manifest Delta table not found: {MANIFEST_PATH}")

manifest_schema = (
    spark.read.format("delta").load(MANIFEST_PATH).schema
)
manifest_columns = [field.name for field in manifest_schema.fields]

required_manifest_columns = {
    "dataset", "file_name", "file_path", "batch_id", "status",
    "first_seen", "processed_at", "run_id", "retry_count",
    "error_message", "data_date", "file_size_bytes",
    "content_hash", "last_seen_at", "discovery_run_id",
    "claimed_at", "lease_expires_at", "last_attempt_at",
    "attempt_count", "last_success_at", "status_updated_at",
    "source_record_count", "written_record_count",
    "rejected_record_count", "processed_content_hash",
    "schema_fingerprint", "target_path", "target_delta_version",
    "processing_duration_seconds", "error_type", "last_error_at"
}

missing_manifest_columns = (
    required_manifest_columns - set(manifest_columns)
)

if missing_manifest_columns:
    raise ValueError(
        f"Manifest is missing required columns: "
        f"{sorted(missing_manifest_columns)}"
    )

SCHEMA_FINGERPRINT = hashlib.sha256(
    SCHEMA_DDL.strip().encode("utf-8")
).hexdigest()

manifest_delta = DeltaTable.forPath(spark, MANIFEST_PATH)

run_id = str(uuid.uuid4())

logger.info("Processing run ID: %s", run_id)
logger.info("Dataset: %s", DATASET_NAME)
logger.info("Source system: %s", SOURCE_SYSTEM)
logger.info("Selected candidates: %s", selected_df.count())


# ---------------------------------------------------------
# 2. Helper functions
# ---------------------------------------------------------

def utc_now_naive():
    """Return UTC datetime without timezone information for Spark."""
    return datetime.now(timezone.utc).replace(tzinfo=None)





def get_manifest_rows(file_name):
    return (
        spark.read.format("delta").load(MANIFEST_PATH)
        .filter(
            (F.col("dataset") == DATASET_NAME)
            & (F.col("file_name") == file_name)
        )
        .select(
            "dataset",
            "file_name",
            "file_path",
            "status",
            "content_hash",
            "attempt_count",
            "retry_count",
            "run_id"
        )
        .collect()
    )


def get_current_file_hash(file_path):
    rows = (
        spark.read.format("binaryFile")
        .load(file_path)
        .select(F.sha2(F.col("content"), 256).alias("hash"))
        .collect()
    )

    if len(rows) != 1 or rows[0]["hash"] is None:
        raise RuntimeError(
            f"Could not calculate SHA-256 for {file_path}"
        )

    return rows[0]["hash"]


def add_bronze_metadata(source_df, file_name, file_path, batch_id):
    """Apply the standard metadata profile configured in Cell 1."""

    result = (
        source_df
        .withColumn(
            INGESTED_AT_COL,
            F.lit(utc_now_naive()).cast("timestamp")
        )
        .withColumn(SOURCE_SYSTEM_COL, F.lit(SOURCE_SYSTEM))
        .withColumn(SOURCE_FILE_COL, F.lit(file_name))
        .withColumn(RUN_ID_COL, F.lit(run_id))
        .withColumn(BATCH_ID_COL, F.lit(batch_id))
    )

    if SOURCE_PATH_COL:
        result = result.withColumn(
            SOURCE_PATH_COL, F.lit(file_path)
        )

    if DATASET_COL:
        result = result.withColumn(
            DATASET_COL, F.lit(DATASET_NAME)
        )

    return result


# ---------------------------------------------------------
# 3. Process selected files
# ---------------------------------------------------------

results = []

for candidate in selected_df.orderBy(
    "effective_data_date", "file_name"
).collect():

    file_name = candidate["file_name"]
    file_path = candidate["file_path"]
    actual_hash = candidate["actual_content_hash"]
    selection_reason = candidate["selection_reason"]

    bronze_write_committed = False
    claim_acquired = False
    batch_id = str(uuid.uuid4())
    file_start_time = utc_now_naive()

    source_count = None
    written_count = None
    target_version = None

    try:
        logger.info("Starting file: %s", file_name)

        # -------------------------------------------------
        # 3A. Recheck source content
        # -------------------------------------------------

        fresh_hash = get_current_file_hash(file_path)

        if fresh_hash != actual_hash:
            raise RuntimeError(
                f"{file_name}: source content changed after discovery. "
                "Rerun Cells 2–4."
            )

        # -------------------------------------------------
        # 3B. Register missing manifest row when required
        # -------------------------------------------------

        current_rows = get_manifest_rows(file_name)

        

        if len(current_rows) != 1:
            raise RuntimeError(
                f"{file_name}: expected exactly one manifest row; "
                f"found {len(current_rows)}."
            )

        current = current_rows[0]
        expected_status = current["status"]
        expected_hash = current["content_hash"]

        if expected_status not in {"NEW", "FAILED", "SUCCESS"}:
            raise RuntimeError(
                f"{file_name}: status {expected_status!r} "
                "is not eligible for processing."
            )

        if selection_reason == "CONTENT_CHANGED":
            if expected_hash is None or expected_hash == actual_hash:
                raise RuntimeError(
                    f"{file_name}: content-change selection is no "
                    "longer valid. Rerun reconciliation."
                )
        elif expected_hash != actual_hash:
            raise RuntimeError(
                f"{file_name}: manifest hash differs from source hash. "
                "Rerun reconciliation."
            )

        if selection_reason == "SUCCESS_REPROCESS_CANDIDATE":
            if RUN_MODE == "INCREMENTAL":
                raise RuntimeError(
                    "Successful-file reprocessing is not allowed "
                    "in INCREMENTAL mode."
                )

            if RUN_MODE == "BACKFILL" and not REPROCESS_SUCCESSFUL:
                raise RuntimeError(
                    "Successful-file reprocessing is disabled "
                    "for this backfill."
                )

        # -------------------------------------------------
        # 3C. Claim the file
        # DeltaTable.update() executes directly; no .execute()
        # -------------------------------------------------

        claim_time = utc_now_naive()
        lease_expiry = claim_time + timedelta(minutes=LEASE_MINUTES)

        claim_condition = (
            (F.col("t.dataset") == DATASET_NAME)
            & (F.col("t.file_name") == file_name)
            & (F.col("t.status") == expected_status)
            & F.col("t.content_hash").eqNullSafe(F.lit(expected_hash))
        )

        manifest_delta.alias("t").update(
            condition=claim_condition,
            set={
                "status": F.lit("PROCESSING"),
                "run_id": F.lit(run_id),
                "claimed_at": F.lit(claim_time).cast("timestamp"),
                "lease_expires_at": F.lit(lease_expiry).cast("timestamp"),
                "last_attempt_at": F.lit(claim_time).cast("timestamp"),
                "attempt_count": (
                    F.coalesce(F.col("t.attempt_count"), F.lit(0)) + 1
                ),
                "retry_count": (
                    F.when(
                        F.col("t.status") == "FAILED",
                        F.coalesce(F.col("t.retry_count"), F.lit(0)) + 1
                    ).otherwise(
                        F.coalesce(F.col("t.retry_count"), F.lit(0))
                    )
                ),
                "status_updated_at": F.lit(claim_time).cast("timestamp"),
                "content_hash": F.lit(actual_hash),
                "schema_fingerprint": F.lit(SCHEMA_FINGERPRINT),
                "error_message": F.lit(None).cast("string"),
                "error_type": F.lit(None).cast("string")
            }
        )

        # Check actual ownership before processing the file.
        owner_rows = get_manifest_rows(file_name)

        if len(owner_rows) != 1:
            raise RuntimeError(
                f"{file_name}: manifest row missing after claim."
            )

        owner = owner_rows[0]

        if (
            owner["status"] != "PROCESSING"
            or owner["run_id"] != run_id
            or owner["content_hash"] != actual_hash
        ):
            raise RuntimeError(
                f"{file_name}: claim was not acquired by this run."
            )

        claim_acquired = True

        # -------------------------------------------------
        # 3D. Read source using configured schema
        # -------------------------------------------------

        source_df = (
            spark.read
            .schema(SOURCE_SCHEMA)
            .option("multiLine", "true")
            .option("mode", "FAILFAST")
            .json(file_path)
        )

        source_count = source_df.count()

        if source_count == 0:
            raise ValueError(f"{file_name}: source file is empty.")

        missing_source_columns = (
            set(SOURCE_SCHEMA.fieldNames()) - set(source_df.columns)
        )

        if missing_source_columns:
            raise ValueError(
                f"{file_name}: missing source columns: "
                f"{sorted(missing_source_columns)}"
            )

        # Dataset-specific quality rules belong in a separate rules layer.

        # -------------------------------------------------
        # 3E. Add standard Bronze metadata
        # -------------------------------------------------

        replacement_df = add_bronze_metadata(
            source_df=source_df,
            file_name=file_name,
            file_path=file_path,
            batch_id=batch_id
        )

        if replacement_df.filter(
            F.col(SOURCE_FILE_COL).isNull()
        ).limit(1).count() > 0:
            raise ValueError(
                f"{file_name}: source-file metadata contains nulls."
            )

        # -------------------------------------------------
        # 3F. First write or replace this file's rows
        # Spark Connect compatible initial write
        # -------------------------------------------------

        bronze_exists = DeltaTable.isDeltaTable(spark, BRONZE_PATH)

        if not bronze_exists:
            # Inspect the target path before using overwrite.
            # Do not overwrite an existing non-Delta location.
            path_exists = False

            try:
                dbutils.fs.ls(BRONZE_PATH)
                path_exists = True
            except Exception as path_exc:
                error_text_path = str(path_exc).lower()

                missing_path = (
                    "filenotfoundexception" in error_text_path
                    or "path does not exist" in error_text_path
                    or "no such file or directory" in error_text_path
                )

                if not missing_path:
                    raise

            if path_exists:
                raise RuntimeError(
                    f"Bronze path exists but is not a Delta table: "
                    f"{BRONZE_PATH}. Inspect it before writing."
                )

            # Initial creation only, after checking the path is absent.
            (
                replacement_df.write
                .format("delta")
                .mode("overwrite")
                .save(BRONZE_PATH)
            )

        else:
            bronze_before_df = (
                spark.read.format("delta").load(BRONZE_PATH)
            )

            expected_bronze_columns = (
                set(SOURCE_SCHEMA.fieldNames())
                | {
                    INGESTED_AT_COL,
                    SOURCE_SYSTEM_COL,
                    SOURCE_FILE_COL,
                    RUN_ID_COL,
                    BATCH_ID_COL
                }
            )

            missing_bronze_columns = (
                expected_bronze_columns - set(bronze_before_df.columns)
            )

            if missing_bronze_columns:
                raise ValueError(
                    f"Existing Bronze schema is missing columns: "
                    f"{sorted(missing_bronze_columns)}. "
                    "Schema migration is required before writing."
                )

            # Replace only this file's records.
            # Assumes filenames are unique within this dataset.
            escaped_file_name = file_name.replace("'", "''")

            (
                replacement_df.write
                .format("delta")
                .mode("overwrite")
                .option(
                    "replaceWhere",
                    f"{SOURCE_FILE_COL} = '{escaped_file_name}'"
                )
                .save(BRONZE_PATH)
            )

        # The write returned successfully.
        bronze_write_committed = True

        # -------------------------------------------------
        # 3G. Verify Bronze write
        # -------------------------------------------------

        post_target = (
            spark.read.format("delta").load(BRONZE_PATH)
            .filter(F.col(SOURCE_FILE_COL) == file_name)
        )

        written_count = post_target.count()

        wrong_run = post_target.filter(
            F.col(RUN_ID_COL) != run_id
        ).count()

        wrong_batch = post_target.filter(
            F.col(BATCH_ID_COL) != batch_id
        ).count()

        wrong_source_system = post_target.filter(
            F.col(SOURCE_SYSTEM_COL) != SOURCE_SYSTEM
        ).count()

        if written_count != source_count:
            raise RuntimeError(
                f"{file_name}: source count {source_count} differs "
                f"from written count {written_count}."
            )

        if wrong_run or wrong_batch or wrong_source_system:
            raise RuntimeError(
                f"{file_name}: post-write metadata validation failed. "
                f"wrong_run={wrong_run}, wrong_batch={wrong_batch}, "
                f"wrong_source_system={wrong_source_system}"
            )

        target_version = (
            DeltaTable.forPath(spark, BRONZE_PATH)
            .history(1)
            .select("version")
            .first()["version"]
        )

        # -------------------------------------------------
        # 3H. Finalize manifest as SUCCESS
        # No .execute() after DeltaTable.update()
        # -------------------------------------------------

        finalized_at = utc_now_naive()
        processing_seconds = (
            finalized_at - file_start_time
        ).total_seconds()

        manifest_delta.alias("t").update(
            condition=(
                (F.col("t.dataset") == DATASET_NAME)
                & (F.col("t.file_name") == file_name)
                & (F.col("t.status") == "PROCESSING")
                & (F.col("t.run_id") == run_id)
                & (F.col("t.content_hash") == actual_hash)
            ),
            set={
                "status": F.lit("SUCCESS"),
                "batch_id": F.lit(batch_id),
                "processed_at": F.lit(finalized_at).cast("timestamp"),
                "last_success_at": F.lit(finalized_at).cast("timestamp"),
                "status_updated_at": F.lit(finalized_at).cast("timestamp"),
                "processed_content_hash": F.lit(actual_hash),
                "schema_fingerprint": F.lit(SCHEMA_FINGERPRINT),
                "source_record_count": F.lit(source_count).cast("long"),
                "written_record_count": F.lit(written_count).cast("long"),
                "rejected_record_count": F.lit(0).cast("long"),
                "target_path": F.lit(BRONZE_PATH),
                "target_delta_version": F.lit(target_version).cast("long"),
                "processing_duration_seconds": F.lit(
                    processing_seconds
                ).cast("double"),
                "error_message": F.lit(None).cast("string"),
                "error_type": F.lit(None).cast("string"),
                "last_error_at": F.lit(None).cast("timestamp"),
                "lease_expires_at": F.lit(None).cast("timestamp"),
                "claimed_at": F.lit(None).cast("timestamp")
            }
        )

        final_rows = get_manifest_rows(file_name)

        if len(final_rows) != 1:
            raise RuntimeError(
                f"{file_name}: final manifest row missing."
            )

        final_row = final_rows[0]

        if (
            final_row["status"] != "SUCCESS"
            or final_row["run_id"] != run_id
        ):
            raise RuntimeError(
                f"{file_name}: manifest finalization was not confirmed."
            )

        results.append((
            file_name,
            "SUCCESS",
            source_count,
            written_count,
            target_version,
            ""
        ))

        logger.info(
            "SUCCESS %s | source=%s | written=%s | Delta version=%s",
            file_name,
            source_count,
            written_count,
            target_version
        )

    except Exception as exc:
        failed_at = utc_now_naive()
        error_text = str(exc)[:4000]

        # Only change the manifest if this run owns the claim.
        # If Bronze committed, preserve a reconciliation warning.
        if claim_acquired:
            try:
                current_owner = (
                    spark.read.format("delta").load(MANIFEST_PATH)
                    .filter(
                        (F.col("dataset") == DATASET_NAME)
                        & (F.col("file_name") == file_name)
                    )
                    .select("status", "run_id")
                    .first()
                )

                if (
                    current_owner
                    and current_owner["status"] == "PROCESSING"
                    and current_owner["run_id"] == run_id
                ):
                    if bronze_write_committed:
                        error_text = (
                            "BRONZE_WRITE_COMMITTED_BUT_FINALIZATION_FAILED; "
                            "reconcile Bronze and manifest before retry. "
                            + error_text
                        )[:4000]

                    manifest_delta.alias("t").update(
                        condition=(
                            (F.col("t.dataset") == DATASET_NAME)
                            & (F.col("t.file_name") == file_name)
                            & (F.col("t.status") == "PROCESSING")
                            & (F.col("t.run_id") == run_id)
                        ),
                        set={
                            "status": F.lit("FAILED"),
                            "status_updated_at": F.lit(
                                failed_at
                            ).cast("timestamp"),
                            "last_error_at": F.lit(
                                failed_at
                            ).cast("timestamp"),
                            "error_message": F.lit(error_text),
                            "error_type": F.lit(type(exc).__name__),
                            "lease_expires_at": F.lit(None).cast("timestamp"),
                            "claimed_at": F.lit(None).cast("timestamp")
                        }
                    )

            except Exception as manifest_exc:
                logger.error(
                    "Could not update failure state for %s: %s",
                    file_name,
                    manifest_exc
                )

        state = (
            "FAILED_AFTER_BRONZE_WRITE_RECONCILIATION_REQUIRED"
            if bronze_write_committed
            else "FAILED"
        )

        logger.exception("%s %s", state, file_name)

        results.append((
            file_name,
            state,
            source_count,
            written_count,
            target_version,
            error_text
        ))


# ---------------------------------------------------------
# 4. Display run results
# ---------------------------------------------------------

result_schema = (
    "file_name string, status string, source_count long, "
    "written_count long, delta_version long, error_message string"
)

results_df = spark.createDataFrame(results, result_schema)

display(results_df)

print("Dataset:", DATASET_NAME)
print("Source system:", SOURCE_SYSTEM)
print("Run ID:", run_id)
print("Files attempted:", len(results))

failed_results = [
    row for row in results
    if row[1] != "SUCCESS"
]

print("Failed files:", len(failed_results))

if any(
    row[1] == "FAILED_AFTER_BRONZE_WRITE_RECONCILIATION_REQUIRED"
    for row in results
):
    print(
        "WARNING: A Bronze write committed but manifest finalization "
        "failed. Reconcile the affected file before retrying."
    )

logger.info("Universal Bronze Cell 5 completed.")
